##**Mastering the Pinecone vector database.**

##**GOAL: save text sentence embeddings and perform semantic search in Pinecone using Python code.**

Create a Pinecone Account: Head to 'Pinecone.io' and sign up for a free account. After signing up, you’ll be given an API key.

 To start working with Pinecone,
install 'pinecone' library

In [1]:
%pip install pinecone sentence-transformers python-dotenv

Note: you may need to restart the kernel to use updated packages.


From 'pinecone' library import Pinecone, ServerlessSpec.

In [2]:
import os
import time
from pathlib import Path
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

c:\Users\jelisejs.bumazkins\Documents\GitHub\Bootcamp_Accenture_JB\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


On the Pinecone main page, navigate to the "Get started" section via the left menu. Next, follow the "Quick Start for Developers" guide.
* Initialaze a client.
* Create an index.
* Upsert data.
* Perform a search.









When you perform "Upsert data", add a list of your own text strings.

When you perform "Search", use of your own text string.

In [4]:
load_dotenv(Path.cwd() / ".env", override=True)
api_key = os.getenv("PINECONE_API_KEY")
assert api_key, "PINECONE_API_KEY not found"

pc = Pinecone(api_key=api_key)

index_name = "bootcamp-semantic-search"

# Create an index if it doesn't exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# Wait until the index is ready
while not pc.describe_index(index_name).status["ready"]:
    time.sleep(2)

index = pc.Index(index_name)
print("Index is ready!")

# Create embeddings from original text strings
model = SentenceTransformer(
    "sentence-transformers/paraphrase-MiniLM-L6-v2"
)

texts = [
    "Banks use artificial intelligence to detect suspicious payments.",
    "A fraud detection system identifies unusual credit card transactions.",
    "Students use mobile applications to manage their expenses.",
    "Regular exercise helps athletes improve their physical condition.",
    "Healthy food and proper sleep support muscle recovery.",
    "University students learn Python to build useful applications.",
    "Teachers use AI tools to create interactive learning materials.",
    "Travelers check weather forecasts before planning outdoor trips."
]

embeddings = model.encode(texts)

# Upsert data
vectors = [
    {
        "id": f"text-{i}",
        "values": embedding.tolist(),
        "metadata": {"text": text}
    }
    for i, (text, embedding) in enumerate(zip(texts, embeddings))
]

index.upsert(vectors=vectors, namespace="bootcamp")
print("Uploaded:", len(vectors), "vectors")

# Semantic search
query = "How can banks identify fraudulent card payments?"
query_vector = model.encode(query).tolist()

time.sleep(5)

results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True,
    namespace="bootcamp"
)

print("Search completed!")

Index is ready!


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1615.11it/s]


Uploaded: 8 vectors
Search completed!


Print search results.

In [5]:
print("Search query:", query)
print("\nTop 3 results:")

for match in results.matches:
    print(f"\nSimilarity: {match.score:.4f}")
    print("Text:", match.metadata["text"])

Search query: How can banks identify fraudulent card payments?

Top 3 results:

Similarity: 0.7663
Text: A fraud detection system identifies unusual credit card transactions.

Similarity: 0.6309
Text: Banks use artificial intelligence to detect suspicious payments.

Similarity: 0.1187
Text: Students use mobile applications to manage their expenses.
